# Laboration 1

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
from torch.nn.utils.rnn import pad_sequence
from collections import defaultdict
import nltk # new
nltk.download('stopwords') # new
nltk.download('punkt_tab')
def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)                      # remove emails
    data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
    data['Sentence'] = data['Sentence'].str.replace(r'[^\w\s]','')                                                       # remove special characters
    data['Sentence'] = data['Sentence'].replace(r'\d', '', regex=True)                                                   # remove numbers
    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if not w in stopwords.words('english')]
        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }
    return data

# If this is the primary file that is executed (ie not an import of another file)
# if __name__ == "__main__":
# get data, pre-process and split
data = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
data.columns = ['Sentence', 'Class']
data['index'] = data.index                                          # add new column index
columns = ['index', 'Class', 'Sentence']
data = preprocess_pandas(data, columns)                             # pre-process
training_data, validation_data, training_labels, validation_labels = train_test_split( # split the data into training, validation, and test splits
    data['Sentence'].values.astype('U'),
    data['Class'].values.astype('int32'),
    test_size=0.10,
    random_state=0,
    shuffle=True
)

# vectorize data using TFIDF and transform for PyTorch for scalability
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')
training_data = word_vectorizer.fit_transform(training_data)        # transform texts to sparse matrix
training_data = training_data.todense()                             # convert to dense matrix for Pytorch
vocab_size = len(word_vectorizer.vocabulary_)
validation_data = word_vectorizer.transform(validation_data)
validation_data = validation_data.todense()
train_x_tensor = torch.from_numpy(np.array(training_data)).type(torch.FloatTensor)
train_y_tensor = torch.from_numpy(np.array(training_labels)).long()
validation_x_tensor = torch.from_numpy(np.array(validation_data)).type(torch.FloatTensor)
validation_y_tensor = torch.from_numpy(np.array(validation_labels)).long()

# def load_data(filepath):
#     data = pd.read_csv(filepath, delimiter='\t', header=None)
#     data.columns = ['Sentence', 'Class']
#     data['index'] = data.index
#     columns = ['index', 'Class', 'Sentence']
#     data = preprocess_pandas(data, columns)
#     training_data, validation_data, training_labels, validation_labels = train_test_split(
#         data['Sentence'].values.astype('U'), # Unicode string
#         data['Class'].values.astype(np.int32),
#         test_size=0.15,
#         shuffle=True
#     )

# # Build vocabulary
#     word_to_idx = defaultdict(lambda: 0)  # 0 will be padding
#     idx = 1
#     for sentence in training_data:
#         for word in sentence.split():
#             if word not in word_to_idx:
#                 word_to_idx[word] = idx
#                 idx += 1
#     vocab_size = len(word_to_idx) + 1  # +1 for padding index 0

#     # Convert sentences to sequences of word indices
#     def sentences_to_indices(sentences, word_to_idx):
#         seqs = []
#         for sentence in sentences:
#             seq = [word_to_idx[word] for word in sentence.split() if word in word_to_idx]
#             seqs.append(torch.tensor(seq, dtype=torch.long))
#         return pad_sequence(seqs, batch_first=True, padding_value=0)

#     train_x_tensor = sentences_to_indices(training_data, word_to_idx)
#     val_x_tensor   = sentences_to_indices(validation_data, word_to_idx)

#     train_y_tensor = torch.tensor(training_labels, dtype=torch.long)
#     val_y_tensor   = torch.tensor(validation_labels, dtype=torch.long)

#     return train_x_tensor, train_y_tensor, val_x_tensor, val_y_tensor, vocab_size





    #     word_vectorizer = TfidfVectorizer( # Text till siffror
    #     analyzer='word',
    #     ngram_range=(1,2), # rangen är mellan ett ord och tvåpar
    #     max_features=50000,
    #     max_df=0.5, # Tar bort ord som finns i över 50% av alla texter, i engelskan är det exempelvis orden: This, The, Is, a och and.
    #     use_idf=True,
    #     norm='l2' # Normaliserar vektorerna
    # )
    # training_data = word_vectorizer.fit_transform(training_data).todense()
    # validation_data = word_vectorizer.transform(validation_data).todense()
    # train_x = torch.from_numpy(np.array(training_data)).float() # Gör från NumPy array till PyTorch Tensor. Vi sparar i float [2.0, 0,3. 0,6 etc]
    # train_y = torch.from_numpy(np.array(training_labels)).long() # Vi sparar i long [0, 1, 0 etc]
    # val_x = torch.from_numpy(np.array(validation_data)).float()
    # val_y = torch.from_numpy(np.array(validation_labels)).long()
    # return train_x, train_y, val_x, val_y

## imports

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, DataLoader
from torch.utils.data import random_split

from importlib import reload
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertModel

import wandb

#from importlib import reload
#import data_loading_code
#reload(data_loading_code)
#from data_loading_code import *


## Load Data

Might be wise to not load the 25K file if it isn't needed then only load the regular one.

In [ ]:
train_dataset_LSTM = TensorDataset(train_x_tensor, train_y_tensor)
train_loader_LSTM = DataLoader(train_dataset_LSTM, batch_size=32, shuffle=True)

val_dataset_LSTM = TensorDataset(validation_x_tensor, validation_y_tensor)
val_loader_LSTM = DataLoader(val_dataset_LSTM, batch_size=32)

In [ ]:
input_size = train_x_tensor.shape[1]
print(input_size)

class LSTM(nn.Module):
    def __init__(self, hidden_size = 128):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first= True)
        self.dropout = nn.Dropout(0.15)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, features)
        out, (hidden, cell) = self.lstm(x)
        return self.fc(hidden[-1])

# Task 1.1
A simple neural network composed of linear layers. You may incorporate activation
functions, dropout, and other complementary layers as needed.

In [ ]:
# settings
epochs = 1 #10
lr=0.0001

# scheduler settings
patience = 3
factor = 0.1

# wandb settings
ys_train = []
ys_valid = []
keys = ["Training loss", "Validation loss"]

### 1.1 ANN: Running model

In [ ]:
model_1_1 = nn.Sequential(
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Dropout(0.15),
    nn.Linear(128,2) # Negative or positive review
)

criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.SGD(model_LSTM.parameters(), lr=0.0001)
optimizer = torch.optim.Adam(model_1_1.parameters(), lr=lr)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor = factor,
        patience = patience # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')

# initialize wandb
wandb.init(project="Lab 1", name="ANN")

for epoch in range(epochs):
    model_1_1.train()

    train_running_loss = 0.0
    for batch_x, batch_y in train_loader_LSTM:

#------------------------- TRAINING -------------------------
        optimizer.zero_grad()
        output = model_1_1(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        train_running_loss+=loss.item()
    average_train_loss = train_running_loss/len(train_loader_LSTM)
    ys_train.append(average_train_loss)

#------------------------- VALIDATION -------------------------
    model_1_1.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader_LSTM:
            output = model_1_1(batch_x)
            val_loss = criterion(output, batch_y)
            val_running_loss += val_loss.item()
        average_val_loss = val_running_loss/len(val_loader_LSTM)
    ys_valid.append(average_val_loss)
    scheduler.step(average_val_loss)

#------------------------- VISUALIZATION -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_train_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )
#------------------------- SAVING BEST MODEL -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model_1_1.state_dict(), "BestModel_1_1.pth")
        print("Saved The Best Performing Model")

# wandb finalization
wandb.log({
    "ANN": wandb.plot.line_series(
        xs=list(range(epochs)),
        ys=[ys_train, ys_valid],
        keys=keys
    )
})
wandb.finish()

## 1.1 ANN: Test run
The best model is tested

In [ ]:
#Test the model
model_1_1.load_state_dict(torch.load("BestModel_1_1.pth"))
model_1_1.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in val_loader_LSTM:
        output = model_1_1(batch_x)
        val_loss = criterion(output, batch_y)
        test_loss += val_loss.item()

        HighestValue, predicted = torch.max(output, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

average_test_loss = test_loss / len(val_loader_LSTM)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

# Task 1.1 LSTM network
A neural network based on LSTM or bidirectional LSTM (Bi-LSTM) layers.

In [ ]:
# settings
epochs = epochs #50
lr=0.0001

# scheduler settings
patience = 3
factor = 0.1

# wandb settings
ys_train = []
ys_valid = []
keys = ["Training loss", "Validation loss"]

## 1.1 LSTM: Training and Validation
The model is trained, validated and the best model is saved.

In [ ]:


model_LSTM = LSTM()
criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(model_LSTM.parameters(), lr=0.0001)
optimizer = torch.optim.Adam(model_LSTM.parameters(), lr=lr)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor = 0.1,
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss_LSTM = float('inf')

# initialize wandb
wandb.init(project="Lab 1", name="RNN with LSTM")

for epoch in range(epochs):
    model_LSTM.train()

    running_train_loss_LSTM = 0.0
    for batch_x, batch_y in train_loader_LSTM:
#------------------------- TRAINING -------------------------
        optimizer.zero_grad()
        output = model_LSTM(batch_x)
        loss_LSTM = criterion(output, batch_y)
        loss_LSTM.backward()
        optimizer.step()
        running_train_loss_LSTM += loss_LSTM.item()
    average_train_loss_LSTM = running_train_loss_LSTM/len(train_loader_LSTM)
    ys_train.append(average_train_loss)

#------------------------- VALIDATION -------------------------
    model_LSTM.eval()
    val_running_loss_LSTM = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader_LSTM:
            output = model_LSTM(batch_x)
            val_loss_LSTM = criterion(output, batch_y)
            val_running_loss_LSTM += val_loss_LSTM.item()
        average_val_loss_LSTM = val_running_loss_LSTM/len(val_loader_LSTM)
        ys_valid.append(average_val_loss_LSTM)
    scheduler.step(average_val_loss_LSTM)

#------------------------- VISUALIZATION -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_train_loss_LSTM:.3f}, "
        f"val loss: {average_val_loss_LSTM:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )
#------------------------- SAVING BEST MODEL -------------------------
    if average_val_loss_LSTM < best_val_loss_LSTM:
        best_val_loss_LSTM = average_val_loss_LSTM
        torch.save(model_LSTM.state_dict(), "BestModel_LSTM.pth")
        print("Saved The Best Performing Model")


# wandb finalization
wandb.log({
    "RNN with LSTM": wandb.plot.line_series(
        xs=list(range(epochs)),
        ys=[ys_train, ys_valid],
        keys=keys
    )
})
wandb.finish()

## 1.1 LSTM Test run
The best model is tested

In [ ]:
model_LSTM.load_state_dict(torch.load("BestModel_LSTM.pth"))
model_LSTM.eval()

test_loss_LSTM = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in val_loader_LSTM:
        output = model_LSTM(batch_x)
        val_loss_LSTM = criterion(output, batch_y)
        test_loss_LSTM += val_loss_LSTM.item()

        HighestValue, predicted = torch.max(output, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

average_test_loss_LSTM = test_loss_LSTM / len(val_loader_LSTM)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss_LSTM:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

wandb.finish()

# Task 1.2: Transformers Implementation
For this task, you will implement your transformer in PyTorch. 

In [ ]:
# settings
epochs = epochs #50
best_val_loss_bert = float('inf')
dataset = "amazon_cells_labelled_LARGE_25K.txt"
#lr = 0.0001

# scheduler settings
patience = 5
factor = 0.1
patience_counter = 0

# wandb settings
ys_train = []
ys_valid = []
keys = ["Training loss", "Validation loss"]

In [ ]:
df = pd.read_csv(dataset, delimiter='\t', header=None)
sentiment_mapping = {0: 'Negative', 1: 'Positive'}
df['Rating'] = df[1].map(sentiment_mapping)#df['Rating'] = df[1].map(sentiment_mapping)
print('Head: \n')
print(df.head())

print('Rating: \n')
print(df['Rating'].value_counts())

In [ ]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, csv_file, tokenizer, max_length):
        self.dataset = pd.read_csv(csv_file, delimiter='\t', header=None, names=['Sentence', 'Class'])
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_dict = {0: 'Negative', 1: 'Positive'}
    

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        review = self.dataset.iloc[idx, 0]
        sentiment = self.dataset.iloc[idx, 1]
        label = self.label_dict[sentiment]

        encoding = self.tokenizer(
            review,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'review_text': review,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(sentiment, dtype=torch.long)
        }

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
review_dataset = ReviewDataset(dataset, tokenizer, 512)
review_dataset[0]
tokenizer.decode(review_dataset[0]['input_ids'])

In [ ]:
train_size_bert = int(0.7 * len(df))
val_size_bert = int(0.15 * len(df))
test_size_bert = int(0.15 * len(df))
train_dataset_bert, val_dataset_bert, test_dataset_bert = random_split(review_dataset, [train_size_bert, val_size_bert, test_size_bert])

train_loader_bert = DataLoader(train_dataset_bert, batch_size=16, shuffle=True)
val_loader_bert = DataLoader(val_dataset_bert, batch_size=16, shuffle=False)
test_loader_bert = DataLoader(test_dataset_bert, batch_size=16, shuffle=False)

print(f"Number of training samples: {len(train_dataset_bert)}")
print(f"Number of validation samples: {len(val_dataset_bert)}")
print(f"Number of test samples: {len(test_dataset_bert)}")

In [ ]:
class CustomDistilBertForClassification(nn.Module):
    def __init__(self, num_labels=2):
        super(CustomDistilBertForClassification, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.pre_classifier = nn.Linear(self.distilbert.config.dim, 64)
        self.dropout = nn.Dropout(0.4)
        self.classifier = nn.Linear(64, num_labels)

    def forward(self, input_ids, attention_mask):
        distilbert_output = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = distilbert_output[0]  # (batch_size, sequence_length, hidden_size)
        pooled_output = hidden_state[:, 0]  # Take the [CLS] token representation
        pooled_output = self.dropout(pooled_output)
        pooled_output = self.pre_classifier(pooled_output)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
model = CustomDistilBertForClassification()

print(model.distilbert)

## 1.2 Transformer: Training and Validation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
# Add weight decay (L2 regularization) to reduce overfitting
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=0.05)

# Learning rate scheduler to reduce LR when validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=patience,
    #verbose=True,
    min_lr=1e-7
)


# initialize wandb
wandb.init(project="Lab 1", name="DistilBERT")

model.train()
for epoch in range(epochs):
    # Training phase
    total_loss = 0
    for batch in train_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader_bert)
    ys_train.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader_bert:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader_bert)
    ys_valid.append(avg_train_loss)
    model.train()
    
    # Step scheduler
    scheduler.step(avg_val_loss)
    
    # Print metrics
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    # Save best model and implement early stopping
    if avg_val_loss < best_val_loss_bert:
        best_val_loss_bert = avg_val_loss
        torch.save(model.state_dict(), "BestModel_DistilBert.pth")
        print(f"Saved best model with validation loss: {avg_val_loss:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

# wandb finalization
wandb.log({
    "DistilBERT": wandb.plot.line_series(
        xs=list(range(epochs)),
        ys=[ys_train, ys_valid],
        keys=keys
    )
})
wandb.finish()

## 1.2 Transformer: Testing

In [ ]:

model.load_state_dict(torch.load("BestModel_DistilBert.pth"))
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        test_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_test_loss = test_loss / len(val_loader_bert)
test_accuracy = 100 * (correct / total)

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Correct Predictions: {correct}/{total}")

## Task 1.3 Comparison
Here, you should compare of all three models; you are requested to use the same test dataset
for Simple ANN, LSTM based model and the Transformer to answer the following:

• Compare the performance of the two models and explain in which scenarios you would
prefer one over the other.

• How did the two models’ complexity, accuracy, and efficiency differ? Did one model
outperform the other in specific scenarios or tasks? If so, why?

• What insights did you obtain concerning data amount to train? Embedding utilized?
Architectural choices made?